# 🇻🇳 Google Colab: Công cụ Cào Quy Mô Lớn (5.000 Video / 10.000 Mẫu), Lọc Cử Động Miệng (Lip-Sync) & Gán Nhãn Đa Phương Thức

Notebook này đã được nâng cấp toàn diện để giải quyết triệt để bài toán chất lượng dữ liệu cảm xúc Tiếng Việt:
1. **Nhận diện Khuôn miệng Người đang nói (`Active Speaker Lip-Sync`)**:
   - Dùng `YuNet ONNX` lấy 5 điểm mốc (`2 mắt, mũi, 2 khóe miệng`) và căn chỉnh khuôn mặt về khung chuẩn `112x112` (loại bỏ hoàn toàn rung lắc đầu/camera).
   - Theo dõi vùng miệng `y ∈ [80..105], x ∈ [36..76]` qua 7 khung hình liên tiếp trong cửa sổ 4 giây và tính tương quan với năng lượng giọng nói. **Chỉ giữ lại mẫu mà người trong ảnh thực sự đang mở miệng nói khớp với âm thanh** (loại bỏ cảnh người đứng nghe hoặc giọng người khác ngoài khung hình).
2. **Gợi ý Nhãn Đa Phương Thức (`Tri-Modal Pseudo-Labeling`)**:
   - **Sắc thái Khuôn mặt**: Chạy mạng `MobileFaceNet FER ONNX` (độ chính xác **88.27%** trên RAF-DB) trực tiếp trên chuỗi mặt người đang nói.
   - **Âm điệu Giọng nói (`Prosody`)**: Phân tích cao độ `Pitch F0`, độ biến thiên ngữ điệu `ΔF0`, trọng tâm phổ tần số (`Spectral Centroid`) và nhịp âm tiết.
   - **Ngữ nghĩa Phụ đề Tiếng Việt**: Đối chiếu hơn **180 từ khóa & cụm từ cảm xúc Tiếng Việt**.
3. **Cào quy mô lớn (`5.000 video raw` $\rightarrow$ `10.000 mẫu processed`)**:
   - Điền sẵn **14 link VTV chuẩn + 105 cụm truy vấn tìm kiếm tự động (`105 × 50 = 5.250 video`)**.
   - Hỗ trợ chế độ `cleanup_raw=True` (xóa file `.mp4` gốc ngay sau khi cắt xong đoạn 4s) giúp cào đủ **10.000 mẫu processed (~1.3 GB)** mà không bị tràn ổ cứng Colab!

## Bước 1: Cài đặt thư viện trên Colab

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!pip uninstall -y -qq opencv-python opencv-python-headless
!pip install -q opencv-python
!pip install -q -U yt-dlp librosa soundfile scikit-learn pandas noisereduce scipy

## Bước 2: Tạo file `youtube_links.txt` (14 Link VTV + 105 Cụm Truy Vấn = 5.264 Nguồn Video)

In [ ]:
%%writefile youtube_links.txt
# ==============================================================================
# DANH SÁCH ĐỀ MỤC, LINK VIDEO & TỪ KHÓA YOUTUBE TIẾNG VIỆT QUY MÔ LỚN
# (MỤC TIÊU: ~5.000 VIDEO RAW -> ~10.000 MẪU PROCESSED CÓ LIP-SYNC KHỚP MIỆNG NÓI)
# ==============================================================================

# [happy]
https://www.youtube.com/watch?v=hVGpm5XOKIU
https://www.youtube.com/watch?v=QFjkkc-m9uw
# [sad]
https://www.youtube.com/watch?v=iXGBS35yGtc
https://www.youtube.com/watch?v=dcXwk2ZZkmw
# [angry]
https://www.youtube.com/watch?v=pUClq17dTSI
https://www.youtube.com/watch?v=vLrgNZxJR94
# [surprise]
https://www.youtube.com/watch?v=1M4WfG2JRds
https://www.youtube.com/watch?v=wzqME4Kv4qA
# [fear]
https://www.youtube.com/watch?v=gLtCHP8Cy1w
https://www.youtube.com/watch?v=jcPI8tfHAv4
# [disgust]
https://www.youtube.com/watch?v=1m0QyZGoVRA
https://www.youtube.com/watch?v=HHtRHnRw5hQ
# [neutral]
https://www.youtube.com/watch?v=2-vlAvTRYJA
https://www.youtube.com/watch?v=JE_ZWBbCraI

# ------------------------------------------------------------------------------
# ĐỀ MỤC 1: [happy] (~750 video)
# ------------------------------------------------------------------------------
ytsearch50:gia đình mình vui bất thình lình trích đoạn hài vui
ytsearch50:phim vtv cảnh vui vẻ hạnh phúc cười tươi
ytsearch50:vợ chồng son phỏng vấn hài hước vui vẻ
ytsearch50:bạn muốn hẹn hò phỏng vấn cười vui
ytsearch50:11 tháng 5 ngày trích đoạn vui vẻ hài hước
ytsearch50:về nhà đi con trích đoạn vui hạnh phúc
ytsearch50:lối nhỏ vào đời trích đoạn vui vẻ
ytsearch50:phim ngắn việt nam kết thúc có hậu hạnh phúc
ytsearch50:phỏng vấn đường phố sài gòn vui vẻ hài hước
ytsearch50:phỏng vấn sinh viên hà nội vui vẻ cười
ytsearch50:anh có phải đàn ông không trích đoạn hài
ytsearch50:nhà trọ balanha trích đoạn hài hước cười
ytsearch50:chúng ta của 8 năm sau cảnh vui vẻ hạnh phúc
ytsearch50:gặp nhau cuối tuần tiểu phẩm hài kịch nói
ytsearch50:kịch nói việt nam cảnh vui mừng hạnh phúc

# ------------------------------------------------------------------------------
# ĐỀ MỤC 2: [sad] (~750 video)
# ------------------------------------------------------------------------------
ytsearch50:thương ngày nắng về trích đoạn khóc xúc động
ytsearch50:hương vị tình thân cảnh bật khóc đau lòng
ytsearch50:về nhà đi con cảnh khóc nước mắt xúc động
ytsearch50:đừng làm mẹ cáu cảnh khóc thương con
ytsearch50:cây táo nở hoa cảnh khóc bi kịch đau khổ
ytsearch50:phim truyền hình vtv cảnh chia tay bật khóc
ytsearch50:điều ước thứ 7 vtv3 câu chuyện xúc động bật khóc
ytsearch50:trạm yêu thương vtv1 xúc động rơi nước mắt
ytsearch50:phim ngắn việt nam cảm động rơi nước mắt
ytsearch50:món quà của cha trích đoạn xúc động khóc
ytsearch50:cuộc đời vẫn đẹp sao cảnh khóc tủi thân
ytsearch50:dưới bóng cây hạnh phúc cảnh buồn khóc
ytsearch50:hành trình công lý cảnh đau buồn nước mắt
ytsearch50:phỏng vấn tâm sự buồn bật khóc tiếng việt
ytsearch50:diễn xuất cảnh khóc casting phim việt nam

# ------------------------------------------------------------------------------
# ĐỀ MỤC 3: [angry] (~750 video)
# ------------------------------------------------------------------------------
ytsearch50:phim vtv cảnh tức giận cãi nhau căng thẳng
ytsearch50:chúng ta của 8 năm sau cảnh tức giận mắng
ytsearch50:về nhà đi con cảnh bố sơn tức giận mắng
ytsearch50:hương vị tình thân cảnh cãi vã căng thẳng
ytsearch50:cây táo nở hoa cảnh tức giận quát tháo
ytsearch50:đừng bắt em phải quên cảnh tức giận căng thẳng
ytsearch50:hướng dương ngược nắng cảnh tranh cãi gay gắt
ytsearch50:hoa hồng trên ngực trái cảnh tức giận mắng
ytsearch50:sống chung với mẹ chồng cảnh cãi nhau căng thẳng
ytsearch50:quỳnh búp bê cảnh tức giận đối đầu kịch tính
ytsearch50:phim truyền hình việt nam cảnh nổi giận quát
ytsearch50:tranh luận căng thẳng chương trình truyền hình việt
ytsearch50:phim ngắn việt nam cảnh tức giận phản bội
ytsearch50:bão ngầm cảnh thẩm vấn căng thẳng tức giận
ytsearch50:đấu trí vtv1 cảnh đối chất căng thẳng

# ------------------------------------------------------------------------------
# ĐỀ MỤC 4: [surprise] (~750 video)
# ------------------------------------------------------------------------------
ytsearch50:đi giữa trời rực rỡ trích đoạn bất ngờ ngạc nhiên
ytsearch50:phim vtv cảnh phát hiện sự thật bất ngờ sững sờ
ytsearch50:gia đình mình vui bất thình lình cảnh bất ngờ
ytsearch50:hương vị tình thân cảnh ngạc nhiên sửng sốt
ytsearch50:ảo thuật đường phố việt nam phản ứng kinh ngạc
ytsearch50:bất ngờ tặng quà phản ứng vỡ òa tiếng việt
ytsearch50:về nhà đi con cảnh phát hiện bất ngờ
ytsearch50:hướng dương ngược nắng cảnh sững sờ bất ngờ
ytsearch50:phim ngắn việt nam cú twist bất ngờ ngạc nhiên
ytsearch50:thương ngày nắng về cảnh ngỡ ngàng phát hiện
ytsearch50:mặt nạ gương cảnh phát hiện bí mật bất ngờ
ytsearch50:biệt dược đen cảnh bất ngờ lộ diện
ytsearch50:nhà trọ balanha cảnh ngạc nhiên hài hước
ytsearch50:11 tháng 5 ngày cảnh bất ngờ sững sờ
ytsearch50:phỏng vấn bất ngờ đường phố việt nam

# ------------------------------------------------------------------------------
# ĐỀ MỤC 5: [fear] (~750 video)
# ------------------------------------------------------------------------------
ytsearch50:biệt dược đen trích đoạn hoảng sợ lo lắng
ytsearch50:mê cung vtv3 cảnh sợ hãi chạy trốn kịch tính
ytsearch50:người phán xử cảnh sợ hãi hoảng loạn
ytsearch50:quỳnh búp bê cảnh sợ hãi cầu cứu
ytsearch50:mặt nạ gương cảnh hoảng sợ kịch tính
ytsearch50:phim hình sự vtv cảnh tội phạm hoảng sợ
ytsearch50:hồ sơ cá sấu cảnh lo sợ nguy hiểm
ytsearch50:bão ngầm cảnh hoảng loạn sợ hãi
ytsearch50:sinh tử vtv1 cảnh lo lắng sợ hãi
ytsearch50:phim kinh dị ngắn việt nam cảnh hoảng sợ
ytsearch50:thử thách nhà ma việt nam sợ hét toáng
ytsearch50:camera giấu kín dọa ma giật mình tiếng việt
ytsearch50:hành trình công lý cảnh hoảng loạn lo sợ
ytsearch50:trói buộc yêu thương cảnh lo lắng sợ hãi
ytsearch50:lựa chọn số phận cảnh nguy hiểm sợ hãi

# ------------------------------------------------------------------------------
# ĐỀ MỤC 6: [disgust] (~750 video)
# ------------------------------------------------------------------------------
ytsearch50:đừng làm mẹ cáu trích đoạn khó chịu chán ghét
ytsearch50:sống chung với mẹ chồng cảnh khó chịu xét nét
ytsearch50:hương vị tình thân cảnh bà xuân khó chịu khinh thường
ytsearch50:cây táo nở hoa cảnh chán ngán mệt mỏi khó chịu
ytsearch50:thương ngày nắng về cảnh mẹ chồng khó chịu
ytsearch50:hoa hồng trên ngực trái cảnh chán ghét khinh bỉ
ytsearch50:dưới bóng cây hạnh phúc cảnh khó chịu phàn nàn
ytsearch50:phim vtv cảnh thái độ khinh thường khó chịu
ytsearch50:review đồ ăn dở phản ứng nhăn mặt tiếng việt
ytsearch50:ăn thử món lạ kinh dị phản ứng rùng mình tiếng việt
ytsearch50:cuộc đời vẫn đẹp sao cảnh khó chịu cằn nhằn
ytsearch50:gia đình mình vui bất thình lình cảnh khó chịu
ytsearch50:trói buộc yêu thương cảnh thái độ gay gắt khó chịu
ytsearch50:tình yêu và tham vọng cảnh khinh thường khó chịu
ytsearch50:phim truyền hình việt nam cảnh ghét bỏ khó chịu

# ------------------------------------------------------------------------------
# ĐỀ MỤC 7: [neutral] (~750 video)
# ------------------------------------------------------------------------------
ytsearch50:bản tin tài chính kinh doanh vtv1 biên tập viên
ytsearch50:điểm tuần vtv24 bình luận điềm tĩnh
ytsearch50:thời sự vtv1 phỏng vấn chuyên gia kinh tế
ytsearch50:cà phê sáng vtv3 trò chuyện khách mời
ytsearch50:vtv2 chất lượng cuộc sống tư vấn bác sĩ
ytsearch50:chuyển động 24h vtv1 dẫn chương trình trường quay
ytsearch50:vtv4 talk vietnam phỏng vấn tiếng việt
ytsearch50:bản tin thời sự htv9 dẫn chương trình rõ mặt
ytsearch50:phỏng vấn chuyên gia giáo dục truyền hình nhân dân
ytsearch50:vtvcab bản tin thể thao dẫn chương trình
ytsearch50:chia sẻ kiến thức sức khỏe bác sĩ nói chuyện
ytsearch50:tọa đàm đối thoại chính sách vtv1
ytsearch50:góc nhìn văn hóa vtv1 biên tập viên dẫn
ytsearch50:vtv3 vui khỏe có ích tư vấn chuyên gia
ytsearch50:bản tin công nghệ vtv2 dẫn chương trình


## Bước 3: Pipeline Căn chỉnh Mặt 5-Landmark, Lọc Cử Động Miệng (Lip-Sync) & Gán Nhãn Đa Phương Thức (MobileFaceNet FER + Âm điệu + Phụ đề)

In [ ]:
import json, re, shutil, subprocess, urllib.request
from dataclasses import dataclass, asdict
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.model_selection import GroupKFold
import yt_dlp

EMOTIONS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
STD_LANDMARKS_112 = np.array([
    [38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366], [41.5493, 92.3655], [70.7299, 92.2041]
], dtype=np.float32)

VI_EMOTION_KEYWORDS = {
    0: ["tức", "giận", "điên", "bực", "ghét", "cáu", "quạo", "vô lý", "chửi", "mắng", "hỗn", "láo", "cút", "im đi", "quá đáng", "phản bội", "bức xúc", "cãi", "hét"],
    1: ["gớm", "kinh tởm", "tởm", "dơ", "bẩn", "ớn", "buồn nôn", "ghê quá", "ngán", "khó chịu", "phiền", "vô duyên", "giả tạo", "coi thường", "khinh"],
    2: ["sợ", "hoảng", "run", "hãi", "ám ảnh", "nguy hiểm", "cứu", "lo lắng", "rùng mình", "chết mất", "ma", "công an", "bắt", "trốn", "cẩn thận", "khiếp", "hoảng loạn"],
    3: ["vui", "cười", "hạnh phúc", "tuyệt", "thích", "yêu", "haha", "hihi", "cảm ơn", "mừng", "đỉnh", "tuyệt vời", "chúc mừng", "đáng yêu", "dễ thương", "tự hào", "ngon", "đẹp quá"],
    4: ["buồn", "khóc", "tiếc", "đau", "thương", "cô đơn", "chán", "tủi", "mệt mỏi", "thất vọng", "chia tay", "mất", "nghèo", "khổ", "tội nghiệp", "nước mắt", "xin lỗi", "hối hận", "đau lòng", "xót", "tổn thương"],
    5: ["ủa", "hả", "trời ơi", "bất ngờ", "ngạc nhiên", "thật á", "không ngờ", "sốc", "wow", "ơ kìa", "sao lại", "thật không", "kinh ngạc", "giật mình"],
    6: ["bình thường", "hôm nay", "bây giờ", "thông tin", "cho biết", "theo đó", "tuy nhiên", "ngoài ra", "cụ thể", "chương trình", "báo cáo", "kế hoạch"],
}

@dataclass
class LabeledSample:
    sample_id: str; video_id: str; speaker: str; start_sec: float; end_sec: float
    face_path: str; audio_path: str; transcript_vi: str; label: int; emotion: str
    confidence: float; reason: str; verified: int = 0; skip: int = 0

def align_face_112(bgr_frame, lm5):
    src, dst = lm5.astype(np.float32), STD_LANDMARKS_112
    tfm, _ = cv2.estimateAffinePartial2D(src, dst, method=cv2.LMEDS)
    if tfm is None:
        tfm, _ = cv2.estimateAffinePartial2D(src, dst)
    return cv2.warpAffine(bgr_frame, tfm, (112, 112), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def align_upper_face_gray_112(gray_frame, lm5):
    tfm, _ = cv2.estimateAffinePartial2D(lm5[:3].astype(np.float32), STD_LANDMARKS_112[:3])
    return cv2.warpAffine(gray_frame, tfm, (112, 112), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def infer_fer_probs_112(fer_net, bgr_list):
    if not bgr_list or fer_net is None:
        return np.ones(7, dtype=np.float32) / 7.0
    probs_accum = np.zeros(7, dtype=np.float32)
    zoo_to_proj = [0, 1, 2, 3, 6, 4, 5]
    raf_priors = np.array([705.0, 717.0, 281.0, 4772.0, 2524.0, 1982.0, 1290.0], dtype=np.float32)
    raf_log_bias = np.log(raf_priors / raf_priors.mean())
    for bgr in bgr_list:
        rgb = (cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0 - 0.5) / 0.5
        fer_net.setInput(cv2.dnn.blobFromImage(rgb), "data")
        logits = fer_net.forward(["label"])[0][0].astype(np.float32) - 0.90 * raf_log_bias
        exp_l = np.exp((logits - np.max(logits)) / 1.25)
        p_zoo = exp_l / (np.sum(exp_l) + 1e-8)
        for z_i, p_i in enumerate(zoo_to_proj):
            probs_accum[p_i] += float(p_zoo[z_i])
    return probs_accum / (probs_accum.sum() + 1e-8)

def compute_mouth_speaking_activity(aligned_grays, audio_sub_rms):
    if len(aligned_grays) < 3:
        return False, 0.0
    up_patches = [g[18:70, 22:90].astype(np.float32) for g in aligned_grays]
    mouth_patches = [g[82:104, 38:74].astype(np.float32) for g in aligned_grays]
    inner_stds = [float(g[84:100, 42:70].astype(np.float32).std()) for g in aligned_grays]
    up_diffs = [float(np.mean(np.abs(up_patches[i+1] - up_patches[i]))) for i in range(len(up_patches)-1)]
    m_diffs = [float(np.mean(np.abs(mouth_patches[i+1] - mouth_patches[i]))) for i in range(len(mouth_patches)-1)]
    mean_up, mean_m, max_m = float(np.mean(up_diffs)), float(np.mean(m_diffs)), float(np.max(m_diffs))
    ratio = mean_m / (mean_up + 1.5)
    std_osc = float(np.max(inner_stds) - np.min(inner_stds))
    open_arr = np.asarray(inner_stds, dtype=np.float32)
    corr = 0.0
    if len(audio_sub_rms) == len(open_arr) and open_arr.std() > 0.5 and audio_sub_rms.std() > 1e-5:
        c = np.corrcoef(open_arr, audio_sub_rms)[0, 1]
        if not np.isnan(c): corr = float(c)
    lip_score = (mean_m / 12.0) * (0.7 + 0.3 * ratio) * (1.0 + 0.25 * max(-0.2, corr))
    ok = (mean_m >= 7.2 and max_m >= 11.0 and ratio >= 1.04 and std_osc >= 1.8 and lip_score >= 0.55)
    return ok, round(lip_score, 4)

def estimate_pitch_f0(y, sr=16000):
    flen, hlen = int(0.04 * sr), int(0.02 * sr)
    min_l, max_l = int(sr / 400.0), int(sr / 80.0)
    f0s, r_th = [], max(0.015, float(np.sqrt(np.mean(y ** 2))) * 0.65)
    for s in range(0, len(y) - flen, hlen):
        fr = y[s:s+flen]
        if float(np.sqrt(np.mean(fr ** 2))) < r_th: continue
        fr = fr - np.mean(fr)
        c = np.correlate(fr, fr, mode="full")[flen-1:]
        if c[0] <= 1e-6: continue
        c = c / c[0]
        sub = c[min_l:max_l]
        if len(sub) == 0: continue
        pk = int(np.argmax(sub)) + min_l
        if c[pk] >= 0.32: f0s.append(float(sr / pk))
    if len(f0s) < 3: return 165.0, 20.0
    arr = np.asarray(f0s, dtype=np.float32)
    return float(np.median(arr)), float(np.std(arr))


## Bước 4: Chạy Cào Streaming (5.000 Video / 10.000 Mẫu), Lọc Lip-Sync & Xuất Splits

In [ ]:
# Có thể chỉnh TARGET_VIDEOS=5000, TARGET_CLIPS=10000 để cào toàn bộ kho dữ liệu lớn trên Colab
TARGET_VIDEOS = 5000
TARGET_CLIPS = 10000
CLEANUP_RAW_MP4 = True  # Tự động xóa file .mp4 sau khi cắt xong để không tràn ổ cứng Colab

yunet_p = "/tmp/face_detection_yunet_2023mar.onnx"
fer_p = "/tmp/fer_mobilefacenet.onnx"
if not Path(yunet_p).exists():
    urllib.request.urlretrieve("https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx", yunet_p)
if not Path(fer_p).exists():
    urllib.request.urlretrieve("https://github.com/opencv/opencv_zoo/raw/main/models/facial_expression_recognition/facial_expression_recognition_mobilefacenet_2022july.onnx", fer_p)
print("✅ Đã tải sẵn mô hình YuNet 5-Landmark & MobileFaceNet FER (88.27% RAF-DB)!")

# Nếu chạy trong repo đã clone, gọi trực tiếp CLI chuẩn của project:
if Path("tools/yt_crawler/__main__.py").exists():
    !python -m tools.yt_crawler crawl --links-file youtube_links.txt --target-videos {TARGET_VIDEOS} --target-clips {TARGET_CLIPS} --cleanup-raw
    shutil.make_archive("vn_youtube_dataset", "zip", ".", "data")
    print("📦 Đã đóng gói vn_youtube_dataset.zip thành công!")